# SAGE — synthetic trajectory generation on Colab

Runs `HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive` (Q4_K_M GGUF) via `llama-cpp-python` with full GPU offload. Generates conversation-level safety training data across every (category, polarity) that SAGE's prompts cover, and writes a pending JSONL per bucket to Drive. You human-review locally afterward and merge with the existing `synthetic.cli merge` command.

**Target hardware:** A100 40GB (Colab Pro+ usually, Pro sometimes). Will also run on L4 24GB with smaller context / lower throughput.

**Before running:**
1. **Runtime → Change runtime type → GPU** (A100 or L4)
2. Mount Drive in the first cell so outputs persist
3. Expect the model download to take ~20 min the first time (~20 GB GGUF)

## 1. Mount Drive

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/sage'
PENDING_DIR = f'{WORK_DIR}/synthetic/pending'
MODEL_CACHE = f'{WORK_DIR}/models'
os.makedirs(PENDING_DIR, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)
print(f'work dir:     {WORK_DIR}')
print(f'pending dir:  {PENDING_DIR}')
print(f'model cache:  {MODEL_CACHE}')

## 2. GPU sanity check

In [ ]:
!nvidia-smi

## 3. Clone the SAGE repo

We reuse the prompt templates and JSONL schema from the main repo so any prompt tweaks propagate automatically.

In [ ]:
%cd /content
!rm -rf sage
!git clone https://github.com/LettuceAI/sage.git
%cd sage
import sys
sys.path.insert(0, '/content/sage')

## 4. Install llama-cpp-python with CUDA wheels

The prebuilt CUDA wheels ship with `CMAKE_ARGS="-DGGML_CUDA=on"` already baked in — no compile step. Choose the wheel matching Colab's CUDA (usually 12.x).

In [ ]:
!pip install -q huggingface_hub
# CUDA 12.x prebuilt wheels (Colab ships 12.1+); change to cu118 if your runtime is older.
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 5. Download the Q4_K_M GGUF

~20 GB first time; cached on Drive afterward so subsequent sessions just mmap it.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO = 'HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive'

# Find the Q4_K_M file — community repos sometimes use Q4_K_M.gguf, q4_k_m.gguf,
# or variants like *-Q4_K_M-00001-of-NN.gguf. Grab the first match.
files = list_repo_files(REPO)
q4km_files = sorted(f for f in files if 'q4_k_m' in f.lower() and f.lower().endswith('.gguf'))
print('Q4_K_M candidates in repo:')
for f in q4km_files:
    print(' ', f)
assert q4km_files, 'no Q4_K_M files found in repo — check naming'
GGUF_FILE = q4km_files[0]
print(f'\nselected: {GGUF_FILE}')

gguf_path = hf_hub_download(
    repo_id=REPO,
    filename=GGUF_FILE,
    cache_dir=MODEL_CACHE,
)
print(f'\nmodel at: {gguf_path}')
print(f"size: {os.path.getsize(gguf_path) / 1e9:.1f} GB")

## 6. Load the model with full GPU offload

MoE 35B-A3B Q4_K_M weights are ~20 GB; full offload fits A100 40GB with huge margin. On L4 24GB it fits tightly — lower `n_ctx` to 2048 if you see OOM.

In [ ]:
from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=gguf_path,
    n_ctx=4096,         # synthesis prompts + output fit comfortably
    n_gpu_layers=-1,    # offload ALL layers to GPU
    n_batch=512,        # prompt batch size
    use_mmap=True,
    use_mlock=False,
    verbose=False,
    seed=42,
)
print(f'loaded in {time.time() - t0:.1f}s')

## 7. Wire the model into SAGE's Generator interface

SAGE's `SyntheticBuilder` expects a `Generator` with a `generate(system, user, ...)` method. Here is a thin wrapper over `Llama.create_chat_completion`.

In [ ]:
from training.data.synthetic.generator import Generator

class LlamaCppGenerator(Generator):
    def __init__(self, llm: Llama):
        self.llm = llm

    def generate(
        self, system: str, user: str, *, max_tokens: int = 2048, temperature: float = 0.9
    ) -> str:
        resp = self.llm.create_chat_completion(
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': user},
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=0.95,
        )
        return resp['choices'][0]['message']['content']

generator = LlamaCppGenerator(llm)

# Sanity check — should produce JSON-looking output
sample = generator.generate(
    system='You are a helpful assistant. Reply in JSON.',
    user='Return a JSON array of three colors.',
    max_tokens=100,
    temperature=0.3,
)
print('sample output:')
print(sample)

## 8. Run synthesis across every (category, polarity)

Uses `SyntheticBuilder` which pulls the prompt templates from `training/data/synthetic/prompts.py`. Output goes to Drive as pending JSONL — every row tagged `meta.review_status = "pending"` so the later `merge` CLI only accepts reviewer-approved rows.

In [ ]:
import json
from pathlib import Path
from training.data.synthetic.builder import SyntheticBuilder
from sage.schema import CATEGORIES

# How many examples per (category, polarity). Tune as you need.
# 500 is a sweet spot — enough signal without drowning reviewers.
PER_BUCKET = 500

# Buckets covered by existing prompts. sexual_minors is POLICY-RESTRICTED
# to negatives only — matches SAGE's safety policy.
BUCKETS: list[tuple[str, str]] = [
    ('grooming',      'positive'),
    ('grooming',      'negative'),
    ('self_harm',     'positive'),
    ('self_harm',     'negative'),
    ('sexual_minors', 'negative'),
]

builder = SyntheticBuilder(generator)

def _serialize(ex):
    return {
        'conversation': ex.conversation.to_dict(),
        'labels': {c.value: ex.labels.get(c, 0.0) for c in CATEGORIES},
        'source': ex.source,
        'meta': ex.meta,
    }

for category, polarity in BUCKETS:
    print(f'\n=== {category} / {polarity} ===')
    t0 = time.time()
    batch = builder.build(
        category=category,
        polarity=polarity,
        n=PER_BUCKET,
        batch_size=5,          # 5 conversations per LLM call; keeps context bounded
        max_tokens=3500,
        temperature=0.95,
    )
    elapsed = time.time() - t0
    out_path = Path(PENDING_DIR) / f'{category}_{polarity}.jsonl'
    with out_path.open('w', encoding='utf-8') as f:
        for ex in batch.examples:
            f.write(json.dumps(_serialize(ex), ensure_ascii=False) + '\n')
    print(f'  wrote {len(batch.examples)} examples in {elapsed:.0f}s  ({elapsed / max(1, len(batch.examples)):.2f}s/ex)')
    if batch.errors:
        print(f'  {len(batch.errors)} errors (first few):')
        for e in batch.errors[:3]:
            print(f'    - {e}')
    print(f'  → {out_path}')

## 9. Preview a handful of generated examples

In [ ]:
import json
from pathlib import Path

for jsonl in sorted(Path(PENDING_DIR).glob('*.jsonl')):
    print(f'\n--- {jsonl.name} ---')
    with jsonl.open() as f:
        for i, line in enumerate(f):
            if i >= 2: break
            row = json.loads(line)
            turns = row['conversation']['turns']
            labels = {k: v for k, v in row['labels'].items() if v > 0}
            print(f'  Example {i + 1}  labels={labels}  turns={len(turns)}')
            for t in turns:
                text = t['text'] if len(t['text']) < 120 else t['text'][:117] + '...'
                print(f'    {t["role"]:<6}: {text}')

## 10. Next steps — do this on your local machine

Download the JSONL files under `/content/drive/MyDrive/sage/synthetic/pending/` to your local `data/synthetic/pending/` directory, then:

1. **Human review** — open each file and flip `meta.review_status` from `"pending"` to `"approved"` or `"rejected"` per row. Reject anything that looks templated, off-topic, or poorly framed. Expect to keep ~70–85% of output from this model.
2. **Merge approved rows**:
   ```bash
   python -m training.data.synthetic.cli merge \
       --in "data/synthetic/pending/*.jsonl" \
       --out data/processed/synthetic.jsonl
   ```
3. **Concatenate into training set**:
   ```bash
   cat data/processed/synthetic.jsonl >> data/processed/train.jsonl
   ```
4. **Re-trajectorize** (so the new synthetic rows also go through context augmentation):
   ```bash
   python -m training.data.trajectorize_cli --in data/processed/train.jsonl --out data/processed/train.traj.jsonl
   ```
5. **Stage 2 fine-tune** from the existing v1 checkpoint at 10× lower learning rates:
   ```bash
   python -m training.train \
       --train data/processed/train.traj.jsonl \
       --val   data/processed/val.traj.jsonl \
       --out   checkpoints/sage-v1.1 \
       --resume-from checkpoints/sage-v1/best.pt \
       --epochs 1 \
       --lr-head 2e-5 --lr-top 8e-6 --lr-mid 3e-6 --lr-bot 1e-6 \
       --batch-size 4 --grad-accum 8 --max-length 256
   ```

That's v1.1.